In [14]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

In [15]:
# Load Titanic dataset
titanic = fetch_openml(name="titanic", version=1, as_frame=True)

# Convert to DataFrame
df = titanic.frame

# Display first 5 rows
df.head()

C:\Users\kvafs\anaconda3\Lib\site-packages\sklearn\datasets\_openml.py:1002: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [16]:
# Shape
print("Shape:", df.shape)

# Columns
print("\nColumns:")
print(df.columns)

# Dataset information
print("\nInfo:")
df.info()

# Summary statistics
print("\nSummary:")
print(df.describe(include="all"))

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())


Shape: (1309, 14)

Columns:
Index(['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket',
       'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   float64 
 1   survived   1309 non-null   category
 2   name       1309 non-null   object  
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   float64 
 6   parch      1309 non-null   float64 
 7   ticket     1309 non-null   object  
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    object  
 10  embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(3), float64(6), ob

In [17]:
print("Shape before removing duplicates:", df.shape)

# Remove duplicate rows
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape before removing duplicates: (1309, 14)
Shape after removing duplicates: (1309, 14)


In [18]:
# Fill numerical columns
df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())

# Fill categorical columns
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["sex"] = df["sex"].fillna(df["sex"].mode()[0])

# Check missing values
print(df.isnull().sum())

pclass          0
survived        0
name            0
sex             0
age             0
sibsp           0
parch           0
ticket          0
fare            0
cabin        1014
embarked        0
boat          823
body         1188
home.dest     564
dtype: int64


In [19]:
# Convert target column to integer
df["survived"] = df["survived"].astype(int)

# Create Age Group
df["Age_Group"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

# Family Size
df["Family_Size"] = df["sibsp"] + df["parch"] + 1

# Fare Per Person
df["Fare_Per_Person"] = df["fare"] / df["Family_Size"]

df.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest,Age_Group,Family_Size,Fare_Per_Person
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO",Young Adult,1.0,211.3375
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON",Child,4.0,37.8875
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON",Child,4.0,37.8875
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON",Young Adult,4.0,37.8875
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON",Young Adult,4.0,37.8875


In [20]:
grouped = df.groupby(["sex", "pclass"]).agg(
    Passenger_Count=("survived", "count"),
    Survival_Rate=("survived", "mean"),
    Average_Age=("age", "mean"),
    Average_Fare=("fare", "mean")
)

print(grouped)

               Passenger_Count  Survival_Rate  Average_Age  Average_Fare
sex    pclass                                                           
female 1.0                 144       0.965278    36.347222    109.412385
       2.0                 106       0.886792    27.513365     23.234827
       3.0                 216       0.490741    23.908179     15.324250
male   1.0                 179       0.340782    38.991155     69.888385
       2.0                 171       0.146199    30.601364     19.904946
       3.0                 493       0.152130    26.557471     12.419597


In [21]:
# Convert selected columns to NumPy array
arr = df[["age", "fare", "Family_Size"]].to_numpy()

print("Shape:", arr.shape)
print("Mean:", np.mean(arr, axis=0))
print("Maximum:", np.max(arr, axis=0))
print("Minimum:", np.min(arr, axis=0))
print("Standard Deviation:", np.std(arr, axis=0))

Shape: (1309, 3)
Mean: [29.50318312 33.28108564  1.88388083]
Maximum: [ 80.     512.3292  11.    ]
Minimum: [0.1667 0.     1.    ]
Standard Deviation: [12.90031593 51.72173224  1.58303407]


In [22]:
print(df.head())

print("\nFinal Shape:", df.shape)

   pclass  survived                                             name     sex  \
0     1.0         1                    Allen, Miss. Elisabeth Walton  female   
1     1.0         1                   Allison, Master. Hudson Trevor    male   
2     1.0         0                     Allison, Miss. Helen Loraine  female   
3     1.0         0             Allison, Mr. Hudson Joshua Creighton    male   
4     1.0         0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female   

       age  sibsp  parch  ticket      fare    cabin embarked boat   body  \
0  29.0000    0.0    0.0   24160  211.3375       B5        S    2    NaN   
1   0.9167    1.0    2.0  113781  151.5500  C22 C26        S   11    NaN   
2   2.0000    1.0    2.0  113781  151.5500  C22 C26        S  NaN    NaN   
3  30.0000    1.0    2.0  113781  151.5500  C22 C26        S  NaN  135.0   
4  25.0000    1.0    2.0  113781  151.5500  C22 C26        S  NaN    NaN   

                         home.dest    Age_Group  Family_Size  